# Catalog agent playground

Run `uv sync --group notebooks` and `uv run jupyter lab` from the repository root, then select the project `.venv` kernel. Set `DATABASE_URL` and `OPENAI_API_KEY` in `.env`. The catalog query requires an existing database populated by the ingestion CLI.

This notebook uses the Python agent directly; no API server is needed. Model calls use your OpenAI account. The agent can list book metadata; it cannot read or summarize passages yet.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import os

# Find the repository whether Jupyter starts here or in notebooks/.
root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "rag_library").is_dir()
)
os.chdir(root)

from rag_library.agent.agent import get_agent, run_agent
from rag_library.agent.context import create_agent_context
from rag_library.catalog import list_books
from rag_library.config import get_settings

## Start fresh

Re-run this cell after editing `.env` or whenever you want a new agent. No checkpointer or server-side history is attached. Each invocation below starts with only the messages you supply.

In [ ]:
get_settings.cache_clear()
available_books = list_books()
display(available_books)
# Set current IDs from the catalog above; one or more books are required.
book_ids = [available_books[0]["id"]] if len(available_books) == 1 else []
context = create_agent_context(book_ids)
agent = get_agent()

## Try a question

Change the message and re-run. This uses the same answer and actual tool-execution records as `POST /agent`.

In [ ]:
question = "What books are available?"
response = run_agent(agent, question, context=context)
print(response.answer)
asdict(response)

## Inspect native agent state

This makes another independent model call. Inspect the messages, tool requests/results, and validated final answer.

In [ ]:
state = agent.invoke({"messages": [{"role": "user", "content": "List the books and their languages."}]}, context=context)
for message in state["messages"]:
    message.pretty_print()

state["structured_response"]

## Continue a conversation explicitly

Pass the previous messages to supply notebook-local history. Re-running the fresh-agent cell and the independent invocation above resets this example.

In [ ]:
state = agent.invoke({
    "messages": [
        *state["messages"],
        {"role": "user", "content": "Which of those books are in Spanish?"},
    ]
}, context=context)
print(state["structured_response"].answer)